# pyGMs BETA: pyTorch & autograd support

To enable the use of pyTorch tensors instead of numpy arrays, set the PYGMS_USE_TORCH environment variable:

In [2]:
import os
os.environ["PYGMS_USE_TORCH"] = "1"

In [3]:
import sys
sys.path.insert(0,'..')   # Override to local development version of pyGMs...
import pyGMs as gm
print(gm)

import torch
import numpy as np
import matplotlib.pyplot as plt

<module 'pyGMs' from '/Users/ihler/Library/CloudStorage/Dropbox/Code/pyGM/notebooks/../pyGMs/__init__.py'>


In [4]:
X = [gm.Var(0,3), gm.Var(1,3), gm.Var(2,2), gm.Var(3,3)]

f = gm.Factor([X[0]], 1.)
f.t

tensor([1., 1., 1.], dtype=torch.float64)

This loads the "factorTorch.py" implementation of the ``Factor()`` class, instead of the "factorNumpy.py" version.

Everything should work basically the same; there are a number of differences in how torch and numpy operate, but I believe they have been harmonized.

### Autograd for factors

In [5]:
# Let's define a factor over X0:
X = [ gm.Var(0,2), gm.Var(1,2) ]
p0 = gm.Factor( [X[0]], [.5,.5] ); 

# Now turn on autograd for this factor:
p0.t.requires_grad_(True)

print(f'Factor {p0} says requires_grad = {p0.requires_grad}')

Factor Factor({0}) says requires_grad = True


### Implicitly defined factors

We do not necessarily want the probabilities (or log probabilities) in our factors to be the parameters we train on.  For example, the distribution $p(X_0)$ is nicely expressed by its two probabilities, but to fit it to data, we would need to perform a constrained optimization problem, enforcing $p(X_0=x)\geq 0$ and $\sum_x p(X_0=x)=1$.  It is often convenient to instead perform an **unconstrained** optimization over some parameters, for example, the "overcomplete exponential family" form.

In this form, we still have one parameter per state $x$ of $X_0$, but we will call these parameters $\theta_x$.  Then, we define $p(X_0=x) = \frac{1}{Z} \exp(\theta_x)$, where $Z=\sum_{x'}\exp(\theta_{x'})$.  Alternatively, we might like to use a "minimal" form, in which we have only a parameter for $X_0=1$, and define $p(X_0=1)=\sigma(\theta_1)=(1+\exp(-\theta_1))^{-1}$, and $p(X_0=0)=1-p(X_0=1)$.

We can accomplish these kinds of representations using implicitly defined factor tables.  To wit, we define a ``FactorModule`` and provide the base parameters and a transformation necessary to produce the probability table for our graphical model:

In [6]:
X = [ gm.Var(0,2), gm.Var(1,2) ]
th0 = gm.Factor( [X[0]], [0.,0.] ); 
th0.t.requires_grad_(True)

p0 = gm.FactorModule(th0, lambda th: th.exp()/th.exp().sum() )  # define p(X0) as exponential family of theta0

Changes (in-place) to $\theta_0$ now propagate through to the transformed values $\rho_0$:

In [7]:
with torch.no_grad():  # can't directly change a grad-enabled entry without no_grad
  th0[1] = 1.
p0.t                   # now p0 is [exp(0) , exp(1)]/Z

tensor([0.2689, 0.7311], dtype=torch.float64, grad_fn=<CloneBackward0>)

### Training using Autograd

We can now construct a `GraphModel` made up of a mix of factors, including basic tensors with or without autograd, and FactorModules which correspond to transformations of underlying parameters.  We may want to avoid copying the input tensors for this setting, since some tensors may be parameters that the user intends to alter.

In [8]:
# Let's define a simple Markov chain over three variables: p(X0)p(X1|X0)p(X2|X1):

X = [ gm.Var(0,2), gm.Var(1,2), gm.Var(2,2) ]

th0 = gm.Factor( [X[0]], [0.,0.] ); th0.t.requires_grad_(True)
p0 = gm.FactorModule(th0, lambda th: th.exp()/th.exp().sum() )

th1g0 = gm.Factor( [X[0],X[1]], [[0.,0.],[0.,0.]]); th1g0.t.requires_grad_(True)
p1g0 = gm.FactorModule( th1g0, lambda th: th.exp()/th.exp().sum([X[1]]) )

th2g1 = gm.Factor( [X[1],X[2]], [[0.,0.],[0.,0.]]); th2g1.t.requires_grad_(True)
p2g1 = gm.FactorModule( th2g1, lambda th: th.exp()/th.exp().sum([X[2]]) )

model = gm.GraphModel([p0,p1g0,p2g1], copy=False)

In [9]:
data = [(0,0,0),(0,0,1),(0,1,0),(0,1,0),(1,0,0),(1,1,0),(1,1,0),(1,1,1)]

We expect the maximum likelihood values of the $\rho_0$, $\rho_{1|0}$, etc., to be equal to the empirical frequencies that we see in the data.  Since maximum likelihood estimation is transformation-invariant, the underlying parameters $\theta_0,\ldots$ should have values that correspond to these probabilities.  The empirical probabilities are:

In [10]:
npData = np.array(data)
print(f'phat(X0) = {npData.mean(0)[0]}')
print(f'phat(X1|X0=0) = {npData[npData[:,0]==0].mean(0)[1]} ; phat(X1|X0=1) = {npData[npData[:,0]==1].mean(0)[1]}')
print(f'phat(X2|X1=0) = {npData[npData[:,1]==0].mean(0)[2]} ; phat(X1|X0=1) = {npData[npData[:,1]==1].mean(0)[2]}')

phat(X0) = 0.5
phat(X1|X0=0) = 0.5 ; phat(X1|X0=1) = 0.75
phat(X2|X1=0) = 0.3333333333333333 ; phat(X1|X0=1) = 0.2


and, if we train using gradient descent on the negative log-likelihood, we find that we do converge to equivalent parameters:

In [11]:
optimizer = torch.optim.SGD(model.parameters(), lr=0.01)

for it in range(400):
    optimizer.zero_grad()                ## Reset gradient accumulation
    loss = -sum(model.logValue( data ))  ## Loss is negative log-likelihood of the observed data
    loss.backward()                      ## Autograd computation
    optimizer.step()                     ## and update underlying parameters

print(p0.t, '\n', p1g0.t, '\n', p2g1.t,'\n')

# Since the model is explicitly normalized, "logValue" is the log-likelihood
print('Log-likelihood of the data set:', float(sum(model.logValue( data )).detach()) )


tensor([0.5000, 0.5000], dtype=torch.float64, grad_fn=<CloneBackward0>) 
 tensor([[0.5000, 0.5000],
        [0.2582, 0.7418]], dtype=torch.float64, grad_fn=<DivBackward0>) 
 tensor([[0.6570, 0.3430],
        [0.7936, 0.2064]], dtype=torch.float64, grad_fn=<DivBackward0>) 

Log-likelihood of the data set: -14.9806187658592


### Training Undirected Models

Undirected models differ from Bayesian networks in that they define an *unnormalized* measure.  To interpret their values as probabilities, we must compute the normalization constant (or partition function) $Z$.  This means that when training using the log-likelihood, we must include $Z$ in the loss -- otherwise, our optimizer will be able to make the unnormalized measure as high as it likes.

First, let's train an undirected model that is Markov-equivalent to the Bayes net Markov chain:

In [12]:
# Let's define a Markov chain using unnormalized factors:

X = [ gm.Var(0,2), gm.Var(1,2), gm.Var(2,2) ]

th01 = gm.Factor( [X[0],X[1]], 0. ); th01.t.requires_grad_(True)
e01 = gm.FactorModule(th01, lambda th: th.exp() )

th12 = gm.Factor( [X[1],X[2]], 0.); th12.t.requires_grad_(True)
e12 = gm.FactorModule( th12, lambda th: th.exp() )

model = gm.GraphModel([e01,e12], copy=False)

We can see that the normalization constant is not one:

In [13]:
print('logZ: ',torch.log(model.joint().sum()))

logZ:  tensor(2.0794, dtype=torch.float64, grad_fn=<LogBackward0>)


In [14]:
optimizer = torch.optim.SGD(model.parameters(), lr=0.01)

for it in range(400):
    optimizer.zero_grad()                ## Reset gradient accumulation
    loss = -sum(model.logValue( data )) + torch.log(model.joint().sum())*len(data)
    loss.backward()                      ## Autograd computation
    optimizer.step()                     ## and update underlying parameters

print(e01.t, '\n', e12.t, '\n')

# Unnormalized model: we need to include the partition function
print('Log-likelihood of the data set:', float(sum(model.logValue( data )).detach())-len(data)*float(torch.log(model.joint().sum()).detach()) )


tensor([[1.2827, 0.8988],
        [0.6433, 1.3483]], dtype=torch.float64, grad_fn=<ExpBackward0>) 
 tensor([[1.2827, 0.6433],
        [2.2001, 0.5508]], dtype=torch.float64, grad_fn=<ExpBackward0>) 

Log-likelihood of the data set: -14.978668186095355


We see that we have learned an equivalent model to the Bayes net chain, with the same log-likelihood, but expressed using unnormalized factors.

Let's now use a pairwise model over all three variables -- this model is not equivalent to any Bayesian network, since it does not possess any table over all three variables jointly, but has no conditional independence:

In [15]:
# Let's define a pairwise Ising-like model over the three variables:

X = [ gm.Var(0,2), gm.Var(1,2), gm.Var(2,2) ]

th01 = gm.Factor( [X[0],X[1]], 0. ); th01.t.requires_grad_(True)
e01 = gm.FactorModule(th01, lambda th: th.exp() )

th12 = gm.Factor( [X[1],X[2]], 0.); th12.t.requires_grad_(True)
e12 = gm.FactorModule( th12, lambda th: th.exp() )

th02 = gm.Factor( [X[0],X[2]], 0); th02.t.requires_grad_(True)
e02 = gm.FactorModule( th02, lambda th: th.exp() )

model = gm.GraphModel([e01,e12,e02], copy=False)

In [16]:
optimizer = torch.optim.SGD(model.parameters(), lr=0.01)

for it in range(400):
    optimizer.zero_grad()                ## Reset gradient accumulation
    loss = -sum(model.logValue( data )) + torch.log(model.joint().sum())*len(data)
    loss.backward()                      ## Autograd computation
    optimizer.step()                     ## and update underlying parameters

print(e01.t, '\n', e12.t, '\n', e02.t,'\n')

# Unnormalized model: we need to include the partition function
print('Log-likelihood of the data set:', float(sum(model.logValue( data )).detach())-len(data)*float(torch.log(model.joint().sum()).detach()) )


tensor([[1.2363, 0.8504],
        [0.6705, 1.4187]], dtype=torch.float64, grad_fn=<ExpBackward0>) 
 tensor([[0.9802, 0.8456],
        [1.7122, 0.7046]], dtype=torch.float64, grad_fn=<ExpBackward0>) 
 tensor([[1.3928, 0.7549],
        [1.2050, 0.7893]], dtype=torch.float64, grad_fn=<ExpBackward0>) 

Log-likelihood of the data set: -14.972129780057056


This model strictly includes the chain model, and indeed we see that it has learned parameters with a higher log-likelihood.  However, the value is only slightly higher, suggesting that the additional parameters are not helping the fit very much beyond the simpler chain graph.

### Pseudo-likelihood

It is difficult to train very large, dense models using the log-likelihood, since the partition function computation can become intractable.  A computationally simple alternative is to use the log-pseudo-likelihood, which measures the probability of each variable given its Markov blanket.  The resulting collection of conditional distributions may not be consistent with any joint distribution, but are far easier to normalize, since each term requires normalization over only a single variable.

In [17]:
# Let's define a pairwise Ising-like model over the three variables:

X = [ gm.Var(0,2), gm.Var(1,2), gm.Var(2,2) ]

th01 = gm.Factor( [X[0],X[1]], 0. ); th01.t.requires_grad_(True)
e01 = gm.FactorModule(th01, lambda th: th.exp() )

th12 = gm.Factor( [X[1],X[2]], 0.); th12.t.requires_grad_(True)
e12 = gm.FactorModule( th12, lambda th: th.exp() )

th02 = gm.Factor( [X[0],X[2]], 0); th02.t.requires_grad_(True)
e02 = gm.FactorModule( th02, lambda th: th.exp() )

model = gm.GraphModel([e01,e12,e02], copy=False)

In [20]:
optimizer = torch.optim.SGD(model.parameters(), lr=0.01)

for it in range(400):
    optimizer.zero_grad()                ## Reset gradient accumulation
    loss = -sum(gm.misc.pseudologlikelihood(model, data ))
    loss.backward()                      ## Autograd computation
    optimizer.step()                     ## and update underlying parameters

print(e01.t, '\n', e12.t, '\n', e02.t,'\n')

# We'll still use the actual log-likelihood, to compare MPLE to the MLE:
print('Log-likelihood of the data set:', float(sum(model.logValue( data )).detach())-len(data)*float(torch.log(model.joint().sum()).detach()) )

tensor([[1.2373, 0.8489],
        [0.6705, 1.4198]], dtype=torch.float64, grad_fn=<ExpBackward0>) 
 tensor([[0.9789, 0.8475],
        [1.7142, 0.7032]], dtype=torch.float64, grad_fn=<ExpBackward0>) 
 tensor([[1.3945, 0.7532],
        [1.2033, 0.7912]], dtype=torch.float64, grad_fn=<ExpBackward0>) 

Log-likelihood of the data set: -14.972115354406201


In this case, at least, the model that we find is just as good as the one that we found via the actual data log-likelihood.